# CH1: LLM 직접 실험 노트북

LLM의 동작 방식을 직접 실험하며 확인합니다.  
Chapter 1에서 배운 이론(토큰 예측, 디코딩 전략, Tool Use)을 코드로 체험하는 시간입니다.

**사용법**: 빈칸(`___`)을 채우고 실행해보세요. 정답은 맨 아래에 있습니다.

**사전 준비**:
- `uv sync` (프로젝트 의존성 설치)
- 환경변수 `OPENROUTER_API_KEY` 설정

In [ ]:
import os
import math
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# LLM 초기화 — OpenRouter를 통해 Gemini 모델을 사용합니다
llm = ChatOpenAI(
    model="google/gemini-3-flash-preview",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

# 연결 확인
test = llm.invoke([HumanMessage(content="안녕하세요, 테스트입니다.")])
print(f"연결 성공: {test.content[:50]}...")

---
### 실험 1: 다음 토큰 확률 분포 직접 보기

Ch1 교재에서 LLM의 본질을 배웠습니다:

> LLM은 입력 토큰들의 통계적 관계를 학습한 가중치를 바탕으로 **다음 토큰을 예측**할 뿐,  
> 사실을 "알고" 답하는 것이 아닙니다.

LLM API의 `logprobs` 옵션을 켜면, 모델이 각 토큰 위치에서 **어떤 후보 토큰을 얼마나 확신했는지** 확률 분포를 직접 볼 수 있습니다.  
"오늘 서울의 날씨는" 뒤에 어떤 토큰이 올지, 모델의 머릿속을 들여다봅니다.

> **참고**: logprobs는 모든 모델/프로바이더가 지원하지는 않습니다. 이 실험에서는 `openai/gpt-4.1-mini`를 사용합니다.

In [ ]:
# logprobs를 지원하는 모델을 사용합니다 (OpenAI 계열)
llm_for_logprobs = ChatOpenAI(
    model="openai/gpt-4.1-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
    model_kwargs={"logprobs": True, "top_logprobs": 5},
)

response = llm_for_logprobs.invoke([HumanMessage(content="오늘 서울의 날씨는")])

print(f"모델 응답: {response.content}\n")
print("=" * 60)
print("각 토큰 위치에서 모델이 고려한 상위 5개 후보:")
print("=" * 60)

logprobs_raw = response.response_metadata.get("logprobs")
logprobs_data = logprobs_raw.get("content", []) if logprobs_raw else []

if not logprobs_data:
    print("\n⚠️ 이 모델은 logprobs를 반환하지 않았습니다.")
    print("   openai/gpt-4.1-mini 등 OpenAI 계열 모델을 사용해 주세요.")
else:
    for i, token_info in enumerate(logprobs_data[:8]):  # 처음 8개 토큰만 표시
        chosen = token_info["token"]
        chosen_prob = math.exp(token_info["logprob"]) * 100
        print(f"\n[위치 {i+1}] 선택된 토큰: '{chosen}' ({chosen_prob:.1f}%)")

        for candidate in token_info.get("top_logprobs", []):
            prob = math.exp(candidate["logprob"]) * 100
            bar = "█" * int(prob / 2)  # 시각적 바 차트
            marker = " ← 선택됨" if candidate["token"] == chosen else ""
            print(f"    '{candidate['token']:>8s}' {prob:5.1f}% {bar}{marker}")

---
### 실험 2: Temperature가 확률 분포에 미치는 영향

> - `temperature=0` → 가장 확률 높은 토큰을 선택 (결정론적)  
> - `temperature=1.0` → 원래 확률 분포대로 샘플링 (적당한 다양성)  
> - `temperature=2.0` → 분포가 평탄해져 낮은 확률 토큰도 선택 (높은 다양성, 오류 위험 증가)

같은 질문을 temperature 0, 1.0, 2.0으로 각각 3번씩 호출해봅니다.  
빈칸을 채우세요.

In [ ]:
question = "오늘 서울의 날씨를 한 문장으로 알려줘"

for temp in [___, ___, ___]:  # 힌트: 0, 1.0, 2.0
    print(f"\n{'='*40}")
    print(f"Temperature = {temp}")
    print('='*40)
    test_llm = ChatOpenAI(
        model="google/gemini-3-flash-preview",
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        temperature=temp,
    )
    for i in range(3):
        response = test_llm.invoke([HumanMessage(content=question)])
        print(f"  시도 {i+1}: {response.content}")

---
### 실험 3: System Prompt의 힘

> 동일한 모델과 도구를 사용해도 **Instructions가 다르면 완전히 다른 Agent**가 됩니다.

같은 질문에 다른 System Prompt를 주면 답변이 완전히 달라집니다.  
빈칸을 채우세요.

In [ ]:
user_question = "메일함에 뭐가 왔어?"

prompts = [
    ___,  # 힌트: "당신은 메일 관리 비서입니다."
    ___,  # 힌트: "당신은 해적입니다. 해적 말투로 답하세요."
    ___,  # 힌트: "당신은 3줄 이내로만 답하는 요약 전문가입니다."
]

for sp in prompts:
    print(f"\n{'='*40}")
    print(f"System: {sp}")
    print('='*40)
    response = llm.invoke([
        SystemMessage(content=sp),
        HumanMessage(content=user_question),
    ])
    print(f"응답: {response.content}")

---
### 실험 4: Tool 정의와 LLM의 선택

> LLM은 Tool의 **docstring(설명)**과 **인자 스키마**를 보고,  
> 사용자의 질문이 Tool의 기능 범위에 해당하는지 판단합니다.

`@tool`로 Tool을 정의하고, LLM이 **언제 Tool을 호출하고 언제 직접 답변하는지** 관찰합니다.  
빈칸을 채우세요.

In [ ]:
from langchain_core.tools import tool

@tool
def check_weather(city: str) -> str:
    """특정 도시의 현재 날씨를 확인합니다.

    Args:
        city: 날씨를 확인할 도시 이름
    """
    return f"{city}의 현재 날씨: 맑음, 15°C"

# LLM에 Tool을 바인딩합니다
llm_with_tools = llm.bind_tools([___])  # 힌트: 위에서 만든 Tool

# Tool이 필요한 질문
response1 = llm_with_tools.invoke([HumanMessage(content="서울 날씨 알려줘")])
print("Tool 필요한 질문:")
print(f"  content: {response1.content}")
print(f"  tool_calls: {response1.tool_calls}")

# Tool이 필요 없는 질문
response2 = llm_with_tools.invoke([HumanMessage(content="안녕하세요")])
print("\nTool 불필요한 질문:")
print(f"  content: {response2.content}")
print(f"  tool_calls: {response2.tool_calls}")

---
### 관찰 정리

**실험 1 — 확률 분포 직접 관찰**
- 모델이 높은 확률로 선택한 토큰과, 2순위·3순위 후보의 확률 차이가 어느 정도였나요?
- 확률이 90% 이상인 위치와 50% 미만인 위치의 차이를 관찰해보세요. 확률이 낮을수록 모델이 "확신 없이 추측"하는 지점이며, 이것이 Hallucination이 발생하기 쉬운 위치입니다.

**실험 2 — Temperature의 효과**
- `temperature=0`일 때 3번의 결과가 동일했나요? (완전한 결정론은 아닙니다)
- `temperature=2.0`일 때 답변의 품질은 어떻게 변했나요?

**실험 3 — System Prompt(Instructions)의 역할**
- 같은 모델이 System Prompt에 따라 얼마나 다르게 동작했나요?
- Ch4의 SKILL.md가 바로 이 Instructions 역할을 합니다.

**실험 4 — Tool 호출의 판단 기준**
- LLM은 어떤 기준으로 Tool 호출 여부를 결정했나요?
- "안녕하세요"에는 `tool_calls`가 비어 있고, "서울 날씨 알려줘"에는 `check_weather` 호출이 포함되었을 것입니다.

---
## 정답

<details>
<summary><b>클릭하여 정답 확인</b></summary>

#### 실험 1 — logprobs
빈칸 없음 (완성 코드). 핵심 포인트:
- `model="openai/gpt-4.1-mini"` — logprobs를 지원하는 모델 사용 (Gemini는 OpenRouter 경유 시 미지원)
- `model_kwargs={"logprobs": True, "top_logprobs": 5}` — 각 위치에서 상위 5개 후보 반환
- `math.exp(logprob)` — 로그 확률을 퍼센트 확률로 변환

#### 실험 2 — Temperature 값
```python
for temp in [0, 1.0, 2.0]:
```

#### 실험 3 — System Prompt
```python
prompts = [
    "당신은 메일 관리 비서입니다.",
    "당신은 해적입니다. 해적 말투로 답하세요.",
    "당신은 3줄 이내로만 답하는 요약 전문가입니다.",
]
```

#### 실험 4 — Tool 바인딩
```python
llm_with_tools = llm.bind_tools([check_weather])
```
`bind_tools()`에 전달하는 것은 **함수 객체** 자체입니다 (문자열이 아님).  
LangChain이 `@tool` 데코레이터로 감싼 함수에서 이름, docstring, 인자 타입을 자동 추출하여 JSON 스키마로 변환합니다.

</details>